In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

torch.manual_seed(47)

Prepare data

In [2]:
bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target

n_sample, n_features = X.shape
print(n_sample, n_features)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)

569 30


Scale data

In [3]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

Convert to Torch

In [5]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [6]:
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

Model

In [31]:
class LogisticRegression(nn.Module):
    def __init__(self, input_features):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.linear(x))

model = LogisticRegression(n_features)

Loss & Optimizer

In [32]:
lr = 0.01
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

Training loop

In [33]:
n_epochs = 500
for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)

    loss.backward()

    optimizer.step()

    optimizer.zero_grad()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch: {epoch+1}, loss = {loss.item()}")

Epoch: 100, loss = 0.26128122210502625
Epoch: 200, loss = 0.194385826587677
Epoch: 300, loss = 0.16544969379901886
Epoch: 400, loss = 0.148519828915596
Epoch: 500, loss = 0.13713829219341278


Evaluation

In [34]:
with torch.no_grad():
    y_pred = model(X_test)
    acc = y_pred.round().eq(y_test).sum()/y_test.numel()
    print(acc)

tensor(0.9825)
